# Multimodal Fake News Detection for Bangla Content
## BanglaBERT + ViT-Base-16 with Gated Fusion

**Pipeline:** Data Exploration → Preprocessing → Feature Extraction → Gated Fusion → Classification → Ablation Study → Evaluation

| Dataset | Samples | Type | Role |
|---|---|---|---|
| MultiBanFakeDetect | 9,600 (balanced, pre-split) | Text + Image | Core — train / val / test |
| Bengali Fake News CSV | 11,900 | Text only | Reference / EDA |
| Multilingual Image Archive | 247,716 | Image only | Reference / EDA |

**Ablation models compared:**
1. Text-only (BanglaBERT)
2. Image-only (ViT)
3. Concat fusion (BanglaBERT + ViT)
4. **Gated fusion — proposed model** (BanglaBERT + ViT)


## 1. Imports & Reproducibility

In [ ]:
%matplotlib inline
import os, re, gc, json, time, warnings, random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, matthews_corrcoef
)

import torch
import torch.nn as nn
# FIX: torch.amp is the non-deprecated path (PyTorch >= 2.0)
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel,
    ViTModel,
    get_linear_schedule_with_warmup
)
from torchvision import transforms

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_style("whitegrid")

print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch       : 2.11.0+cpu
CUDA available: False


## 2. Configuration

In [ ]:
class Config:
    # ── Dataset paths ─────────────────────────────────────────────
    _ROOT = (
        "I:/Sayma Dataset/"
        "MultiBanFakeDetect An Extensive Benchmark Dataset for Multimodal "
        "Under-Resource Bangla Fake News Detection/"
        "MultiBanFakeDetect An Extensive Benchmark Dataset for Multimodal "
        "Under-Resource Bangla Fake News Detection/"
        "MultiBanFakeDetect"
    )
    TRAIN_CSV  = _ROOT + "/Train.csv"
    VAL_CSV    = _ROOT + "/Validation.csv"
    TEST_CSV   = _ROOT + "/Test.csv"
    TRAIN_IMG  = _ROOT + "/Train"
    VAL_IMG    = _ROOT + "/Validation"
    TEST_IMG   = _ROOT + "/Test"
    TEXT_CSV   = (
        "I:/Sayma Dataset/Bengali-Fake-News-Dataset/"
        "Bengali-Fake-News-Dataset 2/bengali_fake_news.csv"
    )
    OUTPUT_DIR = "I:/Sayma Dataset/outputs"

    # ── Models ────────────────────────────────────────────────────
    TEXT_MODEL  = "csebuetnlp/banglabert"
    IMAGE_MODEL = "google/vit-base-patch16-224"

    # ── Hyper-parameters ──────────────────────────────────────────
    MAX_LEN         = 256    # headline + description combined (fits within 512)
    IMG_SIZE        = 224
    BATCH_SIZE      = 16
    WARMUP_EPOCHS   = 3      # Stage 1 — frozen encoders
    FINETUNE_EPOCHS = 7      # Stage 2 — all layers unfrozen
    LR_ENCODER      = 2e-5
    LR_FUSION       = 1e-4
    DROPOUT         = 0.3
    PATIENCE        = 3      # early-stopping patience
    GRAD_ACCUM      = 2      # gradient accumulation (effective batch = BATCH × GRAD_ACCUM)
    USE_AMP         = True   # automatic mixed precision (GPU only)

    SEED   = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# AMP device string required by torch.amp.autocast (non-deprecated API)
_use_amp    = cfg.USE_AMP and torch.cuda.is_available()
_amp_device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
print(f"Device     : {cfg.DEVICE}")
print(f"AMP        : {_use_amp}  (device_type='{_amp_device}')")
print(f"Output dir : {cfg.OUTPUT_DIR}")


Device     : cpu
AMP        : False  (device_type='cpu')
Output dir : I:/Sayma Dataset/outputs


## 3. Data Exploration & Visualization

In [ ]:
train_df = pd.read_csv(cfg.TRAIN_CSV)
val_df   = pd.read_csv(cfg.VAL_CSV)
test_df  = pd.read_csv(cfg.TEST_CSV)

for df in [train_df, val_df, test_df]:
    df["category"] = df["category"].str.strip().replace("busniness", "business")

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("=" * 55)
print(f"  Train      : {len(train_df):,} samples")
print(f"  Validation : {len(val_df):,} samples")
print(f"  Test       : {len(test_df):,} samples")
print(f"  Total      : {len(full_df):,} samples")
print("=" * 55)
print(f"\nColumns    : {list(train_df.columns)}")
print(f"Categories : {sorted(train_df['category'].unique())}")
train_df.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'I:/Sayma Dataset/MultiBanFakeDetect An Extensive Benchmark Dataset for Multimodal Under-Resource Bangla Fake News Detection/MultiBanFakeDetect An Extensive Benchmark Dataset for Multimodal Under-Resource Bangla Fake News Detection/MultiBanFakeDetect/Train.csv'

### 3.1 Image Coverage Check

In [ ]:
def find_image(img_dir, image_id):
    for ext in ["jpg", "jpeg", "png", "webp", "gif"]:
        p = os.path.join(img_dir, f"{image_id}.{ext}")
        if os.path.exists(p):
            return p
    return None

def load_image(path):
    img = Image.open(path)
    if hasattr(img, "n_frames"):   # animated GIF — take first frame
        img.seek(0)
    return img.convert("RGB")

cov_rows = []
for split_name, df, img_dir in [
    ("Train",      train_df, cfg.TRAIN_IMG),
    ("Validation", val_df,   cfg.VAL_IMG),
    ("Test",       test_df,  cfg.TEST_IMG),
]:
    found = sum(1 for iid in df["image_id"] if find_image(img_dir, iid))
    cov_rows.append({
        "Split":    split_name,
        "Total":    len(df),
        "Found":    found,
        "Missing":  len(df) - found,
        "Coverage": f"{found/len(df)*100:.1f}%",
    })

print(pd.DataFrame(cov_rows).set_index("Split").to_string())
print("\n⚠  Missing images → replaced with zero tensors during training.")


### 3.2 Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("MultiBanFakeDetect — Dataset Distributions",
             fontsize=15, fontweight="bold", y=1.01)
PALETTE = {"Fake": "#E63946", "Real": "#2A9D8F"}

# (A) Label per split
ax = axes[0, 0]
split_data = {
    "Train":      [train_df["label"].sum(), (train_df["label"] == 0).sum()],
    "Validation": [val_df["label"].sum(),   (val_df["label"] == 0).sum()],
    "Test":       [test_df["label"].sum(),  (test_df["label"] == 0).sum()],
}
x, w = np.arange(3), 0.35
b1 = ax.bar(x - w/2, [v[0] for v in split_data.values()], w,
            label="Fake", color=PALETTE["Fake"], alpha=0.85)
b2 = ax.bar(x + w/2, [v[1] for v in split_data.values()], w,
            label="Real", color=PALETTE["Real"], alpha=0.85)
ax.bar_label(b1, padding=3, fontsize=9); ax.bar_label(b2, padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["Train", "Validation", "Test"])
ax.set_title("(A) Label Distribution per Split", fontweight="bold")
ax.set_ylabel("Count"); ax.legend()

# (B) Category
ax = axes[0, 1]
cat_counts = full_df.groupby(["category", "label"]).size().unstack(fill_value=0)
cat_counts.columns = ["Real", "Fake"]
cat_counts.sort_values("Fake", ascending=True).plot(
    kind="barh", ax=ax, color=[PALETTE["Real"], PALETTE["Fake"]], alpha=0.85, width=0.7)
ax.set_title("(B) Samples per Category", fontweight="bold")
ax.set_xlabel("Count"); ax.set_ylabel(""); ax.legend(loc="lower right")

# (C) Fake sub-types
ax = axes[1, 0]
tc = train_df["types of fake news"].value_counts()
wedges, texts, ats = ax.pie(
    tc.values, labels=tc.index, autopct="%1.1f%%",
    colors=["#2A9D8F", "#E63946", "#E9C46A", "#F4A261"], startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5))
for at in ats: at.set_fontsize(9)
ax.set_title("(C) Fake News Sub-types (Train)", fontweight="bold")

# (D) Text length
ax = axes[1, 1]
full_df["text_len"] = (full_df["headline"].fillna("").str.len() +
                       full_df["description"].fillna("").str.len())
for lbl, col in [(1, PALETTE["Fake"]), (0, PALETTE["Real"])]:
    lens = full_df[full_df["label"] == lbl]["text_len"]
    name = "Fake" if lbl == 1 else "Real"
    ax.hist(lens, bins=50, alpha=0.6, color=col, label=name)
    ax.axvline(lens.median(), color=col, linestyle="--", linewidth=1.5,
               label=f"{name} median: {lens.median():.0f}")
ax.set_title("(D) Combined Text Length (chars)", fontweight="bold")
ax.set_xlabel("Character Count"); ax.set_ylabel("Frequency"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/dataset_distributions.png", bbox_inches="tight")
plt.show()


### 3.3 Fake News Type × Category Heatmap

In [ ]:
fake_train = train_df[train_df["label"] == 1].copy()
ct     = pd.crosstab(fake_train["category"], fake_train["types of fake news"])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Fake News Type Distribution across Categories (Train)",
             fontsize=13, fontweight="bold")

sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0],
            linewidths=0.4, cbar_kws={"label": "Count"})
axes[0].set_title("Counts", fontweight="bold")
axes[0].set_xlabel("Fake Type"); axes[0].set_ylabel("Category")

sns.heatmap(ct_pct, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1],
            linewidths=0.4, cbar_kws={"label": "% within category"})
axes[1].set_title("Row-normalised (%)", fontweight="bold")
axes[1].set_xlabel("Fake Type"); axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/fake_type_category_heatmap.png", bbox_inches="tight")
plt.show()


### 3.4 Sample Images (Fake vs Real)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Sample Images from Train Set", fontsize=14, fontweight="bold")

for row_idx, label_val in enumerate([1, 0]):
    label_name = "FAKE" if label_val == 1 else "REAL"
    color = "#E63946" if label_val == 1 else "#2A9D8F"
    subset = train_df[train_df["label"] == label_val].sample(5, random_state=cfg.SEED)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes[row_idx, col_idx]
        img_path = find_image(cfg.TRAIN_IMG, row["image_id"])
        if img_path:
            try: ax.imshow(load_image(img_path))
            except Exception:
                ax.text(0.5, 0.5, "Load error", ha="center", va="center",
                        transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, "No image", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
        hl = str(row["headline"])
        hl = hl[:45] + "…" if len(hl) > 45 else hl
        ax.set_title(f"{label_name}\n{hl}", fontsize=7.5,
                     color=color, fontweight="bold", pad=3)
        ax.axis("off")

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/sample_images.png", bbox_inches="tight")
plt.show()


### 3.5 Text Statistics

In [ ]:
stats = []
for split_name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    hl_len = df["headline"].fillna("").str.split().str.len()
    ds_len = df["description"].fillna("").str.split().str.len()
    stats.append({
        "Split":                split_name,
        "Headline words (μ)":   f"{hl_len.mean():.1f}",
        "Headline words (max)":  hl_len.max(),
        "Desc. words (μ)":      f"{ds_len.mean():.1f}",
        "Desc. words (max)":    ds_len.max(),
        "Missing headline":     df["headline"].isna().sum(),
        "Missing description":  df["description"].isna().sum(),
    })

pd.DataFrame(stats).set_index("Split")


### 3.6 Tokenisation Length Analysis

In [ ]:
print("Loading tokenizer for length analysis …")
_tmp_tok   = AutoTokenizer.from_pretrained(cfg.TEXT_MODEL)
sample_df  = train_df.sample(min(500, len(train_df)), random_state=cfg.SEED)

def _token_len(row):
    h = str(row["headline"])    if pd.notna(row["headline"])    else ""
    d = str(row["description"]) if pd.notna(row["description"]) else ""
    return len(_tmp_tok(h, d, truncation=False)["input_ids"])

print("Computing token lengths on 500-sample subset …")
token_lens    = sample_df.apply(_token_len, axis=1)
pct_truncated = (token_lens > cfg.MAX_LEN).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lens, bins=40, color="#457B9D", alpha=0.85, edgecolor="white")
ax.axvline(cfg.MAX_LEN, color="#E63946", linewidth=2, linestyle="--",
           label=f"MAX_LEN = {cfg.MAX_LEN}")
ax.axvline(token_lens.median(), color="#2A9D8F", linewidth=2, linestyle="-.",
           label=f"Median = {token_lens.median():.0f}")
ax.text(0.98, 0.95, f"Truncated at MAX_LEN: {pct_truncated:.1f}%",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.6))
ax.set_xlabel("Token Count (no truncation)")
ax.set_ylabel("Frequency")
ax.set_title("Token Length Distribution — Train Sample (n=500)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/token_length_distribution.png", bbox_inches="tight")
plt.show()

print(f"  Min    : {token_lens.min()}")
print(f"  Median : {token_lens.median():.0f}")
print(f"  95th % : {token_lens.quantile(0.95):.0f}")
print(f"  Max    : {token_lens.max()}")
print(f"  Truncated (>{cfg.MAX_LEN}): {pct_truncated:.1f}%")
del _tmp_tok


## 4. Data Preprocessing

In [ ]:
_URL_RE   = re.compile(r"https?://\S+|www\.\S+")
_KEEP_RE  = re.compile(r"[\u0980-\u09FF\u0020-\u007E\s]")
_SPACE_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = _URL_RE.sub(" ", text)
    text = "".join(_KEEP_RE.findall(text))
    return _SPACE_RE.sub(" ", text).strip()

_s = "দেখুন https://example.com এই 😀 খবর!! Breaking: fire 🔥"
print(f"Raw    : {_s}")
print(f"Cleaned: {clean_text(_s)}")


### 4.1 Dataset Class

In [ ]:
class FakeNewsDataset(Dataset):
    """Paired text-image dataset for multimodal fake news detection."""

    _ZERO_IMAGE = torch.zeros(3, cfg.IMG_SIZE, cfg.IMG_SIZE)

    def __init__(self, df, img_dir, tokenizer, transform, max_len=cfg.MAX_LEN):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def _load_image(self, image_id):
        path = find_image(self.img_dir, image_id)
        if path is None:
            return self._ZERO_IMAGE.clone()
        try:
            img = Image.open(path)
            if hasattr(img, "n_frames"):
                img.seek(0)
            return self.transform(img.convert("RGB"))
        except Exception:
            return self._ZERO_IMAGE.clone()

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        headline    = clean_text(str(row.get("headline",    "")))
        description = clean_text(str(row.get("description", "")))
        enc = self.tokenizer(
            headline, description,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   self._load_image(row["image_id"]),
            "label":          torch.tensor(float(row["label"])),
        }


### 4.2 Tokenizer, Transforms & DataLoaders

In [ ]:
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(cfg.TEXT_MODEL)
print(f"  Model     : {cfg.TEXT_MODEL}")
print(f"  Vocab size: {tokenizer.vocab_size:,}")

# ViT-Base-Patch16-224 expects 224×224, normalised mean=0.5 std=0.5
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(cfg.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
eval_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = FakeNewsDataset(train_df, cfg.TRAIN_IMG, tokenizer, train_transform)
val_ds   = FakeNewsDataset(val_df,   cfg.VAL_IMG,   tokenizer, eval_transform)
test_ds  = FakeNewsDataset(test_df,  cfg.TEST_IMG,  tokenizer, eval_transform)

g = torch.Generator(); g.manual_seed(cfg.SEED)
# num_workers=0 required on Windows; increase to 4+ on Linux/Mac for speed
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available(),
                          generator=g)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

batch = next(iter(train_loader))
print(f"\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k:<16}: {tuple(v.shape)}")


## 5. Model Architecture

```
Input Text ──► BanglaBERT ──► CLS (768-d) ──┐
                                              ├──► Gated Fusion (768-d) ──► Classifier ──► p(fake)
Input Image ──► ViT-Base ──► CLS (768-d) ──┘
```

Both encoders output **768-d** — no projection needed. The gate learns a
**per-dimension** soft weight deciding how much each modality contributes.


In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dim  = self.bert.config.hidden_size   # 768

    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state[:, 0]


class ImageEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.dim = self.vit.config.hidden_size   # 768

    def forward(self, pixel_values):
        return self.vit(pixel_values=pixel_values).last_hidden_state[:, 0]


class GatedFusion(nn.Module):
    """
    gate = sigmoid(W · [t ; v])  ∈ (0,1)^dim
    fused = gate ⊙ t + (1 − gate) ⊙ v
    """
    def __init__(self, dim: int):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())

    def forward(self, t, v):
        return self.gate(torch.cat([t, v], dim=-1)) * t +                (1.0 - self.gate(torch.cat([t, v], dim=-1))) * v

    def fuse(self, t, v):
        g = self.gate(torch.cat([t, v], dim=-1))
        return g, g * t + (1.0 - g) * v


class FakeNewsModel(nn.Module):
    FUSION_MODES = {"gated", "concat", "text_only", "image_only"}

    def __init__(self, text_model, image_model,
                 fusion="gated", dropout=cfg.DROPOUT):
        super().__init__()
        assert fusion in self.FUSION_MODES
        self.fusion_type = fusion
        self.text_enc    = TextEncoder(text_model)
        self.image_enc   = ImageEncoder(image_model)
        dim = self.text_enc.dim

        if fusion == "gated":
            self.fusion_layer = GatedFusion(dim)
            clf_in = dim
        elif fusion == "concat":
            clf_in = dim * 2
        else:
            clf_in = dim

        self.classifier = nn.Sequential(
            nn.Linear(clf_in, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, 1),
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        t = self.text_enc(input_ids, attention_mask)
        v = self.image_enc(pixel_values)
        if self.fusion_type == "gated":
            _, f = self.fusion_layer.fuse(t, v)
        elif self.fusion_type == "concat":
            f = torch.cat([t, v], dim=-1)
        elif self.fusion_type == "text_only":
            f = t
        else:
            f = v
        return self.classifier(f).squeeze(-1)

    def freeze_encoders(self):
        for p in list(self.text_enc.parameters()) + list(self.image_enc.parameters()):
            p.requires_grad = False

    def unfreeze_encoders(self):
        for p in list(self.text_enc.parameters()) + list(self.image_enc.parameters()):
            p.requires_grad = True

    def get_gate_weights(self, input_ids, attention_mask, pixel_values):
        """Returns per-sample mean gate value. Preserves train/eval state."""
        assert self.fusion_type == "gated"
        was_training = self.training
        self.eval()
        with torch.no_grad():
            t = self.text_enc(input_ids, attention_mask)
            v = self.image_enc(pixel_values)
            g, _ = self.fusion_layer.fuse(t, v)
        if was_training:
            self.train()
        return g.cpu().numpy()


### 5.1 Parameter Count

In [ ]:
tmp   = FakeNewsModel(cfg.TEXT_MODEL, cfg.IMAGE_MODEL, fusion="gated")
total = sum(p.numel() for p in tmp.parameters())

rows = []
for name, module in [("BanglaBERT",  tmp.text_enc),
                      ("ViT-Base",    tmp.image_enc),
                      ("GatedFusion", tmp.fusion_layer),
                      ("Classifier",  tmp.classifier)]:
    n = sum(p.numel() for p in module.parameters())
    rows.append({"Component": name,
                 "Parameters": f"{n:,}",
                 "Share": f"{n/total*100:.1f}%"})

param_df = pd.DataFrame(rows)
param_df.loc[len(param_df)] = {"Component": "TOTAL",
                                "Parameters": f"{total:,}", "Share": "100%"}
print(param_df.to_string(index=False))
del tmp; gc.collect()


## 6. Training Infrastructure

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def train_epoch(model, loader, optimizer, scheduler, device, scaler=None):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    n_batches = len(loader)
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False), 1):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        lbls = batch["label"].to(device)

        with autocast(device_type=_amp_device, enabled=_use_amp):
            logits = model(ids, mask, imgs)
            loss   = criterion(logits, lbls) / cfg.GRAD_ACCUM

        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # FIX: also step on the last batch to handle len(loader) % GRAD_ACCUM != 0
        if step % cfg.GRAD_ACCUM == 0 or step == n_batches:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * cfg.GRAD_ACCUM
        all_preds.extend((torch.sigmoid(logits) > 0.5).long().cpu().tolist())
        all_labels.extend(lbls.long().cpu().tolist())

    return total_loss / n_batches, f1_score(all_labels, all_preds, zero_division=0)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_probs, all_labels = 0.0, [], [], []

    for batch in tqdm(loader, desc="  eval ", leave=False):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        lbls = batch["label"].to(device)

        with autocast(device_type=_amp_device, enabled=_use_amp):
            logits = model(ids, mask, imgs)
            loss   = criterion(logits, lbls)

        probs = torch.sigmoid(logits)
        total_loss += loss.item()
        all_preds.extend((probs > 0.5).long().cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(lbls.long().cpu().tolist())

    return {
        "loss":        total_loss / len(loader),
        "accuracy":    accuracy_score(all_labels, all_preds),
        "precision":   precision_score(all_labels, all_preds, zero_division=0),
        "recall":      recall_score(all_labels, all_preds, zero_division=0),
        "f1":          f1_score(all_labels, all_preds, zero_division=0),
        "f1_macro":    f1_score(all_labels, all_preds, average="macro",    zero_division=0),
        "f1_weighted": f1_score(all_labels, all_preds, average="weighted", zero_division=0),
        "mcc":         matthews_corrcoef(all_labels, all_preds),
        "auc":         roc_auc_score(all_labels, all_probs),
        "avg_prec":    average_precision_score(all_labels, all_probs),
        "preds":       all_preds,
        "probs":       all_probs,
        "labels":      all_labels,
    }


In [ ]:
def run_experiment(fusion_type: str, verbose: bool = True) -> dict:
    """
    Stage 1: frozen encoders — train fusion + classifier   (warm-up)
    Stage 2: all layers unfrozen, differential LRs        (fine-tune)
    """
    print(f"\n{'='*60}\n  Experiment : {fusion_type.upper()}\n{'='*60}")

    set_seed(cfg.SEED)
    model  = FakeNewsModel(cfg.TEXT_MODEL, cfg.IMAGE_MODEL,
                           fusion=fusion_type).to(cfg.DEVICE)
    scaler = GradScaler(device=_amp_device) if _use_amp else None

    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    best_val_f1, best_state, patience_cnt = 0.0, None, 0
    t0 = time.time()

    # Stage 1: warm-up
    print("\n[Stage 1] Warm-up — encoders frozen")
    model.freeze_encoders()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR_FUSION, weight_decay=1e-2
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 0, len(train_loader) * cfg.WARMUP_EPOCHS)

    for ep in range(cfg.WARMUP_EPOCHS):
        tr_loss, tr_f1 = train_epoch(model, train_loader, optimizer,
                                      scheduler, cfg.DEVICE, scaler)
        vm = evaluate(model, val_loader, cfg.DEVICE)
        history["train_loss"].append(tr_loss); history["val_loss"].append(vm["loss"])
        history["train_f1"].append(tr_f1);    history["val_f1"].append(vm["f1"])
        if verbose:
            print(f"  Ep {ep+1}/{cfg.WARMUP_EPOCHS}  "
                  f"loss={tr_loss:.4f}  val_f1={vm['f1']:.4f}  mcc={vm['mcc']:.4f}")
        if vm["f1"] > best_val_f1:
            best_val_f1, best_state = vm["f1"], deepcopy(model.state_dict())

    # Stage 2: full fine-tuning
    print("\n[Stage 2] Fine-tuning — all layers")
    model.unfreeze_encoders()
    param_groups = [
        {"params": model.text_enc.parameters(),  "lr": cfg.LR_ENCODER},
        {"params": model.image_enc.parameters(), "lr": cfg.LR_ENCODER},
        {"params": model.classifier.parameters(),"lr": cfg.LR_FUSION},
    ]
    if hasattr(model, "fusion_layer"):
        param_groups.append({"params": model.fusion_layer.parameters(),
                              "lr": cfg.LR_FUSION})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-2)
    total_ft  = len(train_loader) * cfg.FINETUNE_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, total_ft // 10, total_ft)

    for ep in range(cfg.FINETUNE_EPOCHS):
        tr_loss, tr_f1 = train_epoch(model, train_loader, optimizer,
                                      scheduler, cfg.DEVICE, scaler)
        vm = evaluate(model, val_loader, cfg.DEVICE)
        history["train_loss"].append(tr_loss); history["val_loss"].append(vm["loss"])
        history["train_f1"].append(tr_f1);    history["val_f1"].append(vm["f1"])
        if verbose:
            print(f"  Ep {ep+1}/{cfg.FINETUNE_EPOCHS}  "
                  f"loss={tr_loss:.4f}  val_f1={vm['f1']:.4f}  mcc={vm['mcc']:.4f}")
        if vm["f1"] > best_val_f1:
            best_val_f1, best_state, patience_cnt = vm["f1"], deepcopy(model.state_dict()), 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.PATIENCE:
                print(f"  Early stop at epoch {ep+1}"); break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    test_m = evaluate(model, test_loader, cfg.DEVICE)
    test_m["train_time_min"] = round(elapsed / 60, 2)

    print(f"\n  ── Test Results ({elapsed/60:.1f} min) ──")
    for k in ["accuracy","precision","recall","f1","f1_macro","mcc","auc","avg_prec"]:
        print(f"  {k:<14}: {test_m[k]:.4f}")

    torch.save(best_state, os.path.join(cfg.OUTPUT_DIR, f"model_{fusion_type}.pt"))
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return {"model": model, "history": history, "test": test_m}


## 7. Ablation Experiments

All four variants trained on **identical** splits with **identical** protocol.

In [ ]:
results = {}
for fusion in ["text_only", "image_only", "concat", "gated"]:
    results[fusion] = run_experiment(fusion)
print("\n✓ All experiments complete.")


## 8. Results & Analysis

### 8.1 Ablation Comparison Table

In [ ]:
LABEL_MAP = {
    "text_only":  "Text-only (BanglaBERT)",
    "image_only": "Image-only (ViT)",
    "concat":     "Concat Fusion",
    "gated":      "Gated Fusion (proposed)",
}
rows = []
for fusion, res in results.items():
    m = res["test"]
    rows.append({
        "Model":       LABEL_MAP[fusion],
        "Accuracy":    round(m["accuracy"],    4),
        "Precision":   round(m["precision"],   4),
        "Recall":      round(m["recall"],      4),
        "F1 (binary)": round(m["f1"],          4),
        "F1 (macro)":  round(m["f1_macro"],    4),
        "F1 (wt.)":    round(m["f1_weighted"], 4),
        "MCC":         round(m["mcc"],         4),
        "AUC-ROC":     round(m["auc"],         4),
        "Avg Prec":    round(m["avg_prec"],    4),
        "Time (min)":  m["train_time_min"],
    })
results_df = pd.DataFrame(rows).set_index("Model")
print("=" * 95)
print(results_df.to_string())
results_df.to_csv(cfg.OUTPUT_DIR + "/ablation_results.csv")
print("\nSaved → ablation_results.csv")


### 8.2 Ablation Bar Chart

In [ ]:
plot_metrics = ["Accuracy", "Precision", "Recall", "F1 (binary)", "MCC", "AUC-ROC"]
key_map = {"Accuracy":"accuracy","Precision":"precision","Recall":"recall",
           "F1 (binary)":"f1","MCC":"mcc","AUC-ROC":"auc"}
bar_colors = ["#A8DADC", "#457B9D", "#E9C46A", "#E63946"]
x, width   = np.arange(len(plot_metrics)), 0.18

fig, ax = plt.subplots(figsize=(16, 6))
for i, (fusion, color) in enumerate(zip(results, bar_colors)):
    vals = [results[fusion]["test"][key_map[m]] for m in plot_metrics]
    bars = ax.bar(x + i*width - 1.5*width, vals, width,
                  label=LABEL_MAP[fusion], color=color, alpha=0.88, edgecolor="white")
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.004,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7, rotation=45)
ax.set_xticks(x); ax.set_xticklabels(plot_metrics, fontsize=10)
ax.set_ylim(0, 1.12); ax.set_ylabel("Score")
ax.set_title("Ablation Study — Test Set Performance", fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=8)
ax.axhline(0.9, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/ablation_bar_chart.png", bbox_inches="tight")
plt.show()


### 8.3 Training Curves (Gated Fusion)

In [ ]:
hist   = results["gated"]["history"]
epochs = range(1, len(hist["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Gated Fusion — Training Curves", fontsize=13, fontweight="bold")

for ax, tr_k, va_k, title, ylabel in [
    (ax1, "train_loss", "val_loss", "Loss",    "BCE Loss"),
    (ax2, "train_f1",  "val_f1",  "F1-Score", "Binary F1"),
]:
    ax.plot(epochs, hist[tr_k], label="Train",      color="#457B9D", linewidth=2)
    ax.plot(epochs, hist[va_k], label="Validation", color="#E63946",
            linewidth=2, linestyle="--", marker="o", markersize=4)
    ax.axvspan(1, cfg.WARMUP_EPOCHS + 0.5, alpha=0.07, color="gray",
               label="Warm-up (frozen)")
    ax.axvline(cfg.WARMUP_EPOCHS + 0.5, color="gray", linestyle=":", linewidth=1.2)
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel); ax.legend(fontsize=9); ax.set_xticks(list(epochs))

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/training_curves.png", bbox_inches="tight")
plt.show()


### 8.4 Confusion Matrices — All Models

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("Confusion Matrices — All Ablation Models (Test Set)",
             fontsize=14, fontweight="bold")

for col, fusion in enumerate(["text_only", "image_only", "concat", "gated"]):
    m  = results[fusion]["test"]
    cm = confusion_matrix(m["labels"], m["preds"])
    for row, (data, fmt, row_title) in enumerate([
        (cm,                                                    "d",    "Counts"),
        (cm.astype(float)/cm.sum(axis=1, keepdims=True), ".2%", "Normalised"),
    ]):
        ax = axes[row, col]
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", ax=ax,
                    xticklabels=["Real","Fake"], yticklabels=["Real","Fake"],
                    linewidths=0.5, cbar=False)
        if row == 0:
            ax.set_title(LABEL_MAP[fusion], fontweight="bold", fontsize=9)
        ax.set_xlabel("Predicted", fontsize=9)
        ax.set_ylabel("Actual" if col == 0 else "", fontsize=9)
        if col == 3:
            ax2 = ax.twinx(); ax2.set_ylabel(row_title, fontsize=9,
                                              rotation=270, labelpad=14)
            ax2.set_yticks([])

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/confusion_matrices_all.png", bbox_inches="tight")
plt.show()


### 8.5 ROC & Precision-Recall Curves

In [ ]:
colors_roc = {"text_only":"#A8DADC","image_only":"#457B9D",
               "concat":"#E9C46A","gated":"#E63946"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("ROC & Precision-Recall Curves — All Models (Test Set)",
             fontsize=13, fontweight="bold")

for fusion, res in results.items():
    m, col = res["test"], colors_roc[fusion]
    fpr, tpr, _   = roc_curve(m["labels"], m["probs"])
    prec, rec, _  = precision_recall_curve(m["labels"], m["probs"])
    ax1.plot(fpr, tpr, linewidth=2, color=col,
             label=f"{LABEL_MAP[fusion]}  (AUC={m['auc']:.4f})")
    ax2.plot(rec, prec, linewidth=2, color=col,
             label=f"{LABEL_MAP[fusion]}  (AP={m['avg_prec']:.4f})")

ax1.plot([0,1],[0,1],"k--",linewidth=1,label="Random")
ax1.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
        title="ROC Curve", xlim=[-0.01,1.01], ylim=[-0.01,1.05])
ax1.legend(loc="lower right", fontsize=8)

ax2.axhline(0.5,color="k",linestyle="--",linewidth=1,label="Random (AP=0.50)")
ax2.set(xlabel="Recall", ylabel="Precision",
        title="Precision-Recall Curve", xlim=[-0.01,1.01], ylim=[0.3,1.05])
ax2.legend(loc="lower left", fontsize=8)

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/roc_pr_curves.png", bbox_inches="tight")
plt.show()


### 8.6 Per-Category F1 Performance

In [ ]:
# Batched inference using IndexedDataset
class IndexedDataset(Dataset):
    def __init__(self, base_ds):
        self.base = base_ds
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        item = self.base[idx]; item["idx"] = idx; return item

best_model = results["gated"]["model"]
idx_loader = DataLoader(IndexedDataset(test_ds), batch_size=cfg.BATCH_SIZE,
                        shuffle=False, num_workers=0)

all_preds_cat, all_labels_cat, all_idx = [], [], []
best_model.eval()
with torch.no_grad():
    for batch in tqdm(idx_loader, desc="per-category eval"):
        ids  = batch["input_ids"].to(cfg.DEVICE)
        mask = batch["attention_mask"].to(cfg.DEVICE)
        imgs = batch["pixel_values"].to(cfg.DEVICE)
        with autocast(device_type=_amp_device, enabled=_use_amp):
            logits = best_model(ids, mask, imgs)
        all_preds_cat.extend((torch.sigmoid(logits) > 0.5).long().cpu().tolist())
        all_labels_cat.extend(batch["label"].long().tolist())
        all_idx.extend(batch["idx"].tolist())

test_reset = test_df.reset_index(drop=True)
# Include fake_type so downstream FP/FN analysis can use it
cat_df = pd.DataFrame({
    "category":  test_reset.loc[all_idx, "category"].values,
    "fake_type": test_reset.loc[all_idx, "types of fake news"].values,
    "label":     all_labels_cat,
    "pred":      all_preds_cat,
})

cat_f1  = cat_df.groupby("category").apply(
    lambda g: f1_score(g["label"], g["pred"], zero_division=0)).reset_index()
cat_acc = cat_df.groupby("category").apply(
    lambda g: accuracy_score(g["label"], g["pred"])).reset_index()
cat_f1.columns  = ["Category", "F1-Score"]
cat_acc.columns = ["Category", "Accuracy"]
cat_summary = cat_f1.merge(cat_acc, on="Category").sort_values("F1-Score").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
bar_cols  = ["#E63946" if v < 0.85 else "#2A9D8F" for v in cat_summary["F1-Score"]]
ax.barh(cat_summary["Category"], cat_summary["F1-Score"], color=bar_cols, alpha=0.85)
ax.axvline(cat_summary["F1-Score"].mean(), color="navy", linestyle="--", linewidth=1.5,
           label=f"Mean F1 = {cat_summary['F1-Score'].mean():.3f}")
ax.set_xlim(0, 1.15); ax.set_xlabel("F1-Score")
ax.set_title("Per-Category F1 — Gated Fusion (Test Set)", fontweight="bold")
ax.legend()

# FIX: O(1) lookup instead of O(n) list.index() inside the loop
cat_pos = {cat: i for i, cat in enumerate(cat_summary["Category"])}
for _, row2 in cat_summary.iterrows():
    ax.text(row2["F1-Score"] + 0.01, cat_pos[row2["Category"]],
            f"F1={row2['F1-Score']:.3f}  Acc={row2['Accuracy']:.3f}",
            va="center", fontsize=8)

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/per_category_f1.png", bbox_inches="tight")
plt.show()
print(cat_summary.to_string(index=False))


### 8.7 False Positive & False Negative Breakdown

In [ ]:
# FP = real news predicted as fake  |  FN = fake news predicted as real
fp_df = cat_df[(cat_df["label"] == 0) & (cat_df["pred"] == 1)].copy()
fn_df = cat_df[(cat_df["label"] == 1) & (cat_df["pred"] == 0)].copy()

print(f"False Positives (Real → Fake): {len(fp_df)}  ({len(fp_df)/len(cat_df)*100:.1f}%)")
print(f"False Negatives (Fake → Real): {len(fn_df)}  ({len(fn_df)/len(cat_df)*100:.1f}%)")
print(f"Total errors                 : {len(fp_df)+len(fn_df)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Error Analysis — Gated Fusion (Test Set)",
             fontsize=13, fontweight="bold")

# (A) FP / FN per category
ax = axes[0]
fp_cat = fp_df["category"].value_counts().sort_index()
fn_cat = fn_df["category"].value_counts().sort_index()
all_cats = sorted(set(fp_cat.index) | set(fn_cat.index))
x_err = np.arange(len(all_cats))
w_err = 0.38
ax.bar(x_err - w_err/2, [fp_cat.get(c,0) for c in all_cats], w_err,
       label="FP (Real→Fake)", color="#E9C46A", alpha=0.9)
ax.bar(x_err + w_err/2, [fn_cat.get(c,0) for c in all_cats], w_err,
       label="FN (Fake→Real)", color="#E63946", alpha=0.9)
ax.set_xticks(x_err); ax.set_xticklabels(all_cats, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Error Count")
ax.set_title("(A) Errors per Category", fontweight="bold")
ax.legend(fontsize=9)

# (B) FN broken down by fake sub-type
ax = axes[1]
fn_types = fn_df["fake_type"].value_counts()
bars = ax.bar(fn_types.index, fn_types.values,
              color=["#E63946","#F4A261","#E9C46A"][:len(fn_types)], alpha=0.85)
ax.bar_label(bars, padding=3)
ax.set_title("(B) False Negatives by Fake Type\n(missed fake news)", fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Fake News Type")

# (C) Error rate per category (proportion of errors in that category)
ax = axes[2]
err_rate = (
    cat_df.groupby("category")
    .apply(lambda g: (g["label"] != g["pred"]).mean() * 100)
    .sort_values(ascending=False)
    .reset_index()
)
err_rate.columns = ["Category", "Error Rate (%)"]
bar_c = ["#E63946" if v > 15 else "#E9C46A" if v > 10 else "#2A9D8F"
         for v in err_rate["Error Rate (%)"]]
ax.barh(err_rate["Category"], err_rate["Error Rate (%)"],
        color=bar_c, alpha=0.85)
ax.axvline(err_rate["Error Rate (%)"].mean(), color="navy", linestyle="--",
           linewidth=1.5, label=f"Mean = {err_rate['Error Rate (%)'].mean():.1f}%")
ax.set_xlabel("Error Rate (%)")
ax.set_title("(C) Error Rate per Category", fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/error_analysis.png", bbox_inches="tight")
plt.show()


### 8.8 Gate Weight Analysis (Gated Fusion)

In [ ]:
gated_model = results["gated"]["model"]
gate_fake, gate_real = [], []

for batch in tqdm(test_loader, desc="gate weights", leave=False):
    ids  = batch["input_ids"].to(cfg.DEVICE)
    mask = batch["attention_mask"].to(cfg.DEVICE)
    imgs = batch["pixel_values"].to(cfg.DEVICE)
    lbls = batch["label"].long().tolist()
    # FIX: consistent autocast with the rest of the pipeline
    with autocast(device_type=_amp_device, enabled=_use_amp):
        g = gated_model.get_gate_weights(ids, mask, imgs)   # (B, 768)
    for gate_vec, lbl in zip(g, lbls):
        (gate_fake if lbl == 1 else gate_real).append(gate_vec.mean())

real_m, fake_m = np.mean(gate_real), np.mean(gate_fake)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Gate Weight Distribution — Text vs Image Dominance",
             fontsize=13, fontweight="bold")

ax = axes[0]
ax.hist(gate_fake, bins=40, alpha=0.7, color="#E63946", label="Fake", density=True)
ax.hist(gate_real, bins=40, alpha=0.7, color="#2A9D8F", label="Real", density=True)
for val, col, name in [(0.5, "k", "Equal"), (real_m, "#2A9D8F", f"Real μ={real_m:.3f}"),
                        (fake_m, "#E63946", f"Fake μ={fake_m:.3f}")]:
    ax.axvline(val, color=col, linestyle="--" if val == 0.5 else "-.", linewidth=1.2,
               label=name)
ax.set_xlabel("Mean Gate Value (→1.0 text, →0.0 image)")
ax.set_ylabel("Density"); ax.set_title("Distribution", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[1]
ax.boxplot([gate_real, gate_fake], labels=["Real", "Fake"], patch_artist=True,
           boxprops=dict(facecolor="#E0F4F1", linewidth=1.5),
           medianprops=dict(color="navy", linewidth=2),
           whiskerprops=dict(linewidth=1.5), capprops=dict(linewidth=1.5))
ax.axhline(0.5, color="k", linestyle="--", linewidth=1, alpha=0.5)
ax.set_ylabel("Mean Gate Value"); ax.set_title("Box Plot", fontweight="bold")
ax.text(1, real_m+0.01, f"μ={real_m:.3f}", ha="center",
        color="#2A9D8F", fontweight="bold")
ax.text(2, fake_m+0.01, f"μ={fake_m:.3f}", ha="center",
        color="#E63946", fontweight="bold")

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/gate_weights.png", bbox_inches="tight")
plt.show()
print(f"Real → {'TEXT' if real_m > 0.5 else 'IMAGE'} dominant  μ={real_m:.4f}")
print(f"Fake → {'TEXT' if fake_m > 0.5 else 'IMAGE'} dominant  μ={fake_m:.4f}")


### 8.9 Sample Predictions

Check this

---





In [1]:
def show_predictions(model, df, img_dir, n=4, correct=True, title_prefix=""):
    model.eval()
    records, tries = [], 0
    for _, row in df.sample(frac=1, random_state=cfg.SEED).iterrows():
        if len(records) >= n or tries >= len(df): break
        tries += 1
        ds_tmp = FakeNewsDataset(pd.DataFrame([row]), img_dir, tokenizer, eval_transform)
        item = ds_tmp[0]
        with torch.no_grad():
            logits = model(
                item["input_ids"].unsqueeze(0).to(cfg.DEVICE),
                item["attention_mask"].unsqueeze(0).to(cfg.DEVICE),
                item["pixel_values"].unsqueeze(0).to(cfg.DEVICE)
            )
            prob = torch.sigmoid(logits).item()
        pred = int(prob > 0.5)
        label = int(row["label"])
        if (pred == label) == correct:
            records.append({"row": row, "prob": prob, "pred": pred, "label": label})

    if not records:
        print("No samples found.")
        return

    fig, axes = plt.subplots(1, len(records), figsize=(5 * len(records), 7))
    if len(records) == 1: axes = [axes]
    fig.suptitle(title_prefix, fontsize=14, fontweight="bold", y=0.98)

    for ax, rec in zip(axes, records):
        img_path = find_image(img_dir, rec["row"]["image_id"])
        if img_path:
            try:
                ax.imshow(load_image(img_path))
            except Exception:
                ax.text(0.5, 0.5, "Image Load Error", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, "No Image", ha="center", va="center", transform=ax.transAxes)

        ax.axis("off")
        gt = "FAKE" if rec["label"] == 1 else "REAL"
        pr = "FAKE" if rec["pred"] == 1 else "REAL"

        headline = str(rec['row']['headline'])
        display_headline = (headline[:40] + "...") if len(headline) > 40 else headline

        title_text = f"{display_headline}\nGT: {gt} | Pred: {pr}\nConf: {rec['prob']:.2%}"
        ax.set_title(title_text, fontsize=10, fontweight="bold",
                     color="#2A9D8F" if rec["pred"] == rec["label"] else "#E63946",
                     pad=20)

    plt.subplots_adjust(top=0.85, bottom=0.1, wspace=0.3)
    plt.show()

# Use the loaded_model initialized in the previous cell
show_predictions(loaded_model, test_df, cfg.TEST_IMG, n=4, correct=True,
                 title_prefix="Correct Predictions — Gated Fusion")
show_predictions(loaded_model, test_df, cfg.TEST_IMG, n=4, correct=False,
                 title_prefix="Incorrect Predictions — Gated Fusion")

NameError: name 'loaded_model' is not defined

### 8.10 Final Metrics Dashboard

In [ ]:
key_map      = {"Accuracy":"accuracy","F1 (binary)":"f1","MCC":"mcc","AUC-ROC":"auc"}
dash_metrics = list(key_map.keys())
bar_colors   = ["#A8DADC","#457B9D","#E9C46A","#E63946"]

fig, axes = plt.subplots(1, len(dash_metrics), figsize=(18, 5))
fig.suptitle("Final Comparison — Gated Fusion vs Baselines",
             fontsize=13, fontweight="bold")

for ax, metric in zip(axes, dash_metrics):
    vals  = [results[f]["test"][key_map[metric]] for f in results]
    bars  = ax.bar(range(len(vals)), vals, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(["Text","Image","Concat","Gated"], fontsize=9)
    margin = 0.06 if metric == "MCC" else 0.04
    ax.set_ylim(max(-0.1, min(vals)-margin), min(1.0, max(vals)+margin+0.06))
    ax.set_title(metric, fontweight="bold"); ax.set_ylabel("Score")
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.004,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    bars[-1].set_edgecolor("darkred"); bars[-1].set_linewidth(2.5)

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR + "/final_dashboard.png", bbox_inches="tight")
plt.show()


## 9. Save Best Model & Results

In [ ]:
all_metrics = {}
for fusion, res in results.items():
    all_metrics[fusion] = {k: round(v,4) for k,v in res["test"].items()
                           if isinstance(v, float)}

with open(cfg.OUTPUT_DIR + "/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("Saved: all_metrics.json")

all_history = {f: res["history"] for f, res in results.items()}
with open(cfg.OUTPUT_DIR + "/training_history.json", "w") as f:
    json.dump(all_history, f, indent=2)
print("Saved: training_history.json")

best_ckpt = {
    "model_state": results["gated"]["model"].state_dict(),
    "fusion":      "gated",
    "text_model":  cfg.TEXT_MODEL,
    "image_model": cfg.IMAGE_MODEL,
    "max_len":     cfg.MAX_LEN,
    "dropout":     cfg.DROPOUT,
    "f1_test":     all_metrics["gated"]["f1"],
    "mcc_test":    all_metrics["gated"]["mcc"],
    "auc_test":    all_metrics["gated"]["auc"],
}
torch.save(best_ckpt, cfg.OUTPUT_DIR + "/best_model_gated.pt")
print("Saved: best_model_gated.pt")

print("\n" + "="*55 + f"\n  All outputs → {cfg.OUTPUT_DIR}\n" + "="*55)
for fname in sorted(os.listdir(cfg.OUTPUT_DIR)):
    fpath = os.path.join(cfg.OUTPUT_DIR, fname)
    sz    = os.path.getsize(fpath)
    unit  = "MB" if sz > 1_000_000 else "KB"
    print(f"  {fname:<42} {sz/(1e6 if unit=='MB' else 1e3):.1f} {unit}")


### 9.1 Load Model from Checkpoint

In [ ]:
# Reload the saved best model without re-running training
# (useful for inference in a fresh session)
def load_model_from_checkpoint(ckpt_path: str, device=cfg.DEVICE) -> FakeNewsModel:
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = FakeNewsModel(
        text_model  = ckpt["text_model"],
        image_model = ckpt["image_model"],
        fusion      = ckpt["fusion"],
        dropout     = ckpt.get("dropout", cfg.DROPOUT),
    ).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded  : {ckpt['fusion']} fusion")
    print(f"F1      : {ckpt['f1_test']}")
    print(f"MCC     : {ckpt['mcc_test']}")
    print(f"AUC     : {ckpt['auc_test']}")
    return model

loaded_model = load_model_from_checkpoint(
    cfg.OUTPUT_DIR + "/best_model_gated.pt"
)


## 10. Inference Demo

In [ ]:
def predict(headline: str, description: str,
            image_path: str = None,
            model=None,
            device=cfg.DEVICE) -> dict:
    """Single-sample inference. Pass model=loaded_model for a fresh-session run."""
    if model is None:
        model = results["gated"]["model"]
    model.eval()

    h, d = clean_text(headline), clean_text(description)
    enc  = tokenizer(h, d, max_length=cfg.MAX_LEN, padding="max_length",
                     truncation=True, return_tensors="pt")

    if image_path and os.path.exists(image_path):
        try:   img = eval_transform(load_image(image_path))
        except Exception: img = torch.zeros(3, cfg.IMG_SIZE, cfg.IMG_SIZE)
    else:
        img = torch.zeros(3, cfg.IMG_SIZE, cfg.IMG_SIZE)

    with torch.no_grad():
        logit = model(enc["input_ids"].to(device),
                      enc["attention_mask"].to(device),
                      img.unsqueeze(0).to(device))
        prob  = torch.sigmoid(logit).item()

    return {
        "prediction":       "FAKE" if prob > 0.5 else "REAL",
        "probability_fake": round(prob, 4),
        "confidence":       f"{max(prob, 1-prob):.2%}",
    }

# ── Demo on first test sample ────────────────────────────────────────────────
sample = test_df.iloc[0]
out    = predict(
    headline    = sample["headline"],
    description = sample["description"],
    image_path  = find_image(cfg.TEST_IMG, sample["image_id"]),
)
print(f"Headline   : {str(sample['headline'])[:80]}")
print(f"True Label : {'FAKE' if sample['label']==1 else 'REAL'}")
print(f"Prediction : {out['prediction']}")
print(f"Confidence : {out['confidence']}")
print(f"P(fake)    : {out['probability_fake']}")
